# 04 — Signal Classification
**Purpose:** Translate 233 German BMS signal names into usable physical quantities.

There is no BMS tag dictionary. We infer meaning from four layers:
1. **Prefix convention** — `V_real`=setpoint, `greal`=calculated, `real`=measured
2. **German compound word decomposition** — `WP`=heat pump, `Abtau`=defrost, `Sek`=seconds
3. **Value range sanity check** — values 0-100 with "Zustand" → likely SOC (%)
4. **Thesis confirmation** — Guillet 2016 Table 25 lists key database variables

Every signal is tagged with which ZORO use cases it supports:
`energy | hvac | heatpump | pv | battery | weather | fdd | mpc`

In [ ]:
import sys, csv
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from zoro_eda.config import load_config
from zoro_eda.paths import resolve_paths
from zoro_eda.classify import classify_signal, SignalClassification

cfg = load_config()
paths = resolve_paths(cfg=cfg)
print("Libraries loaded.")

## Layer 1: Prefix convention

The prefix is the most reliable signal type indicator — before we look
at any other part of the name:

| Prefix | Type | Example |
|---|---|---|
| `V_real` / `Vreal` | Setpoint / config param | `V_real_maxVorlaufTemp` — max supply temp setpoint |
| `greal` | Calculated / aggregated | `greal_LeistungGebaeude` — building power demand |
| `real` | Raw measured value | `real_BatterieLeistung` — battery power measured |
| `grealCluster` | Battery cluster data | `grealCluster_1_Spannung` — cluster 1 voltage |
| `green` | SMA battery cluster raw | `green_bat` — raw cluster current |
| `dint` | Counter / pulse integral | `dintVolGasVerbrauchBhkwImp1` — gas pulse count |

In [ ]:
def get_prefix_type(name: str) -> str:
    n = name.lower()
    if n.startswith("v_real") or n.startswith("vreal"): return "setpoint"
    if n.startswith("grealcluster"):                     return "battery_cluster"
    if n.startswith("greal"):                            return "calculated"
    if n.startswith("real"):                             return "measured"
    if n.startswith("green"):                            return "battery_raw"
    if n.startswith("dint"):                             return "counter"
    return "other"

test_signals = [
    "V_real_maxVorlaufTemp", "greal_BatterieLadeZustand",
    "real_BatterieLeistung", "grealCluster_1_Spannung",
    "green_bat", "dintVolGasVerbrauchBhkwImp1"
]
for sig in test_signals:
    print(f"  {sig:<40} → {get_prefix_type(sig)}")

## Layer 2: German compound word decomposition

Signal names are built from compound German BMS terms.
Here are the key vocabulary items:

| German | English | Example |
|---|---|---|
| WP | heat pump (Wärmepumpe) | `greal_WP1AbtauSek` |
| BHKW | CHP (Blockheizkraftwerk) | `real_P_BHKW` |
| Abtau | defrost | `greal_WP1AbtauSek` |
| Sek | seconds (Sekunden) | suffix on timer signals |
| Vorlauf | supply/flow temperature | `V_real_maxVorlaufTemp` |
| Ruecklauf | return temperature | `greal_W_WMZ_TR_Altbau` |
| Leistung | power | `greal_LeistungGebaeude` |
| Energie | energy | `greal_E_WP` |
| WMZ | heat meter (Wärmemengenzähler) | `greal_K_WMZ_TV_Nord` |
| FBH | underfloor heating | `grealThermLeistungWaermeFBH_Nord` |
| Nachtabsenkung | night setback | `V_real_Nachtabsenkung` |

The classification rules in `src/zoro_eda/signal_rules.py` encode this
vocabulary systematically.

## Layer 3: Classify all 233 signals

In [ ]:
csv_files = sorted([f for f in paths.raw_data.iterdir()
                   if f.is_file() and f.suffix.lower() == ".csv"])

classified = [classify_signal(fpath.name) for fpath in csv_files]

from collections import Counter
cat_counts = Counter(r.category for r in classified)
conf_counts = Counter(r.confidence for r in classified)
excl = sum(1 for r in classified if r.exclude)
unk  = sum(1 for r in classified if r.category == "unknown")
pipe = sum(1 for r in classified if not r.exclude and r.zoro_metric and r.zoro_unit)

print(f"Total: {len(classified)} | Excluded: {excl} | Unknown: {unk} | Pipeline-ready: {pipe}")
print(f"Confidence: {dict(conf_counts)}")
print()
print("Top 10 categories:")
for cat, count in cat_counts.most_common(10):
    print(f"  {cat:<35} {count}")

## Layer 4: ZORO pipeline mapping

Each signal is mapped to the ZORO JSON v1 payload format:
`{timestamp, device_id, metric, value, unit}`

`device_id` is built as: `{building_id_prefix}/{device_id_suffix}`

**Example:**
```json
{
  "timestamp": "2024-02-27T09:12:41Z",
  "device_id":  "de-enfa-main-01/battery-system",
  "metric":     "battery_soc",
  "value":      71.5,
  "unit":       "%"
}
```

In [ ]:
# Show pipeline mappings for the HP FDD signals (our recommended first MVP)
hp_fdd_signals = [r for r in classified
                  if r.use_heatpump and r.use_fdd and not r.exclude]

print(f"Heat Pump FDD signals ({len(hp_fdd_signals)}):")
print(f"{'Signal':<45} {'Metric':<35} {'Unit'}")
print("-" * 90)
for r in sorted(hp_fdd_signals, key=lambda x: x.signal_name):
    print(f"  {r.signal_name:<43} {r.zoro_metric:<35} {r.zoro_unit}")

## Use-case signal coverage

In [ ]:
use_cases = ["energy", "hvac", "heatpump", "pv", "battery", "weather", "fdd", "mpc"]
print("ZORO use-case signal coverage:")
for uc in use_cases:
    count = sum(1 for r in classified if getattr(r, f"use_{uc}"))
    bar = "=" * (count // 3)
    print(f"  {uc:<12} {count:3d}  {bar}")

## Key findings

| Use Case | Signals | Readiness |
|---|---|---|
| HVAC optimization | 129 | Ready |
| Fault detection | 120 | Ready |
| MPC / setpoint control | 109 | Ready |
| Energy analytics | 106 | Ready |
| Heat pump analysis | 60 | **Ready — recommended first MVP** |
| Battery optimization | 23 | Ready |
| PV | 16 | Partially (no irradiance) |
| Weather | 6 | Partially (no irradiance) |

**Confidence in the classification:**
- `high` (197 signals): name + prefix + value range all consistent; thesis confirmed
- `medium` (31 signals): interpretation reasonable, unit uncertain
- `low` (5 signals): genuinely ambiguous — manual review needed

**Important caveat:** Classification is expert judgment from signal names and
value ranges — not validated against the BMS tag list. All `medium` and `low`
confidence signals should be reviewed against the live BMS configuration before
use in production models.

## What to do next

Run the full pipeline:
```bash
python scripts/05_classify_signals.py
```

This writes:
- `reports/signal_classification.csv` — all 233 signals
- `reports/zoro_pipeline_mapping.csv` — 223 pipeline-ready signals with JSON v1 fields
- `context/signal_dictionary.md` — human-readable reference